In [1]:
import torch
import cying.nn as cynn

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F


class OptLangLayer(nn.Module):
    def __init__(
        self,
        d_head,
        n_head,
        hid_width
    ):
        super().__init__()
        self.d_head = d_head
        self.n_head = n_head
        self.d_model = d_head * n_head
        self.hid_width = hid_width

        self.q_linear = nn.Linear(self.d_model, self.d_model)
        self.k_linear = nn.Linear(self.d_model, self.d_model)
        self.v_linear = nn.Linear(self.d_model, self.d_model)
        # self.qk_weight = nn.Parameter(torch.zeros(2, n_head, d_head//2+1))
        # self.theta = nn.Parameter(torch.log(torch.exp(10000**(-torch.arange(d_head//2+1)/(d_head//2+1)))-1).expand(n_head,d_head//2+1))
        self.register_buffer('theta', 10000**(-torch.arange(d_head//2+1)/(d_head//2+1)))
        self.norm_att = nn.LayerNorm(self.d_model)

        self.nonlinear = nn.Sequential(
            nn.Linear(self.d_model, hid_width),
            nn.GELU(),
            nn.Linear(hid_width, self.d_model)
        )

    def forward(
        self,
        token_seq,
        padding_mask=None,
        causal_mask=None
    ):
        token_att = self._multi_head_attention(
            token_seq,
            padding_mask,
            causal_mask
        )
        return self.nonlinear(token_att) + token_seq

    def _multi_head_attention(
        self,
        token_seq,
        padding_mask,
        causal_mask
    ):
        B,L,_= token_seq.shape
        q = self.q_linear(token_seq).reshape([B,L,self.n_head,self.d_head])
        k = self.k_linear(token_seq).reshape([B,L,self.n_head,self.d_head])
        v = self.v_linear(token_seq).reshape([B,L,self.n_head,self.d_head])
        # token_seq_head = token_seq.reshape([B,L,self.n_head,self.d_head])

        s = self._get_score(q, k)

        s = s.masked_fill(
            mask=padding_mask.view((1,B,1,L)),
            value=-float('inf')
        ) if padding_mask is not None else s

        s = s.masked_fill(
            mask=causal_mask,
            value=-float('inf')
        ) if causal_mask is not None else s

        out = torch.matmul(
            input=s.softmax(dim=-1),
            other=v.permute(2,0,1,3)
        ).permute(1,2,0,3).flatten(-2,-1)

        return self.norm_att(out)

    def _get_score(
        self,
        q,
        k
    ):
        times = torch.arange(token_seq.shape[1],device=token_seq.device)[:,None,None]
        pe_weight = torch.exp(1j*times*self.theta[None,None,:])

        # qk_weight = torch.complex(self.qk_weight[0]+1,self.qk_weight[1])

        # token_seq_fre = torch.fft.rfft(token_seq) * pe_weight / self.d_head
        q_fre = torch.fft.rfft(q) * pe_weight / self.d_head
        k_fre = torch.fft.rfft(k) * pe_weight / self.d_head

        s = torch.matmul(
            q_fre.permute(2,0,1,3),
            k_fre.permute(2,0,3,1).conj()
        ).real * 2

        return s

In [18]:
opt_lang_model = torch.load('./models/model18.pth',weights_only=False,map_location='cpu')

In [2]:
from torchinfo import summary
device = torch.device('mps')
num_class = 10**4
opt_lang_model = cynn.OptLangModel(
    num_class,
    512,
    4,
    1024,
    2,
    2
).to(device)
src = torch.randint(0,num_class,[10,15]).to(device)
tgt = torch.randint(0,num_class,[10,32]).to(device)
encoder_padding_mask = torch.randint(0,2,[10,15]).bool().to(device)
encoder_causal_mask = torch.randint(0,2,[10,15,15]).bool().to(device)
decoder_padding_mask = torch.randint(0,2,[10,32]).bool().to(device)
decoder_causal_mask = torch.randint(0,2,[10,32,32]).bool().to(device)
y = opt_lang_model(src,tgt,encoder_padding_mask,decoder_padding_mask,encoder_causal_mask,decoder_causal_mask)
print(y.shape)
summary(opt_lang_model)

torch.Size([10, 32, 10000])


/Users/wangh/Work/Code/PyCying/cying/nn/modules/opt_lang_layer.py:79: UserWarning: An output with one or more elements was resized since it had shape [], which does not match the required output shape [10, 15, 4, 65]. This behavior is deprecated, and in a future PyTorch release outputs will not be resized unless they have zero elements. You can explicitly reuse an out tensor t by resizing it, inplace, to zero elements with t.resize_(0). (Triggered internally at /Users/runner/work/pytorch/pytorch/pytorch/aten/src/ATen/native/Resize.cpp:38.)
  token_seq_fre = torch.fft.rfft(token_seq) / self.d_head
/Users/wangh/Work/Code/PyCying/cying/nn/modules/opt_lang_layer.py:79: UserWarning: An output with one or more elements was resized since it had shape [], which does not match the required output shape [10, 32, 4, 65]. This behavior is deprecated, and in a future PyTorch release outputs will not be resized unless they have zero elements. You can explicitly reuse an out tensor t by resizing it, 

Layer (type:depth-idx)                   Param #
OptLangModel                             524
├─Embedding: 1-1                         5,120,000
├─LayerNorm: 1-2                         1,024
├─ModuleList: 1-3                        --
│    └─OptLangLayer: 2-1                 524
│    │    └─LayerNorm: 3-1               1,024
│    │    └─Sequential: 3-2              1,050,112
│    └─OptLangLayer: 2-2                 524
│    │    └─LayerNorm: 3-3               1,024
│    │    └─Sequential: 3-4              1,050,112
├─ModuleList: 1-4                        --
│    └─OptLangLayer: 2-3                 524
│    │    └─LayerNorm: 3-5               1,024
│    │    └─Sequential: 3-6              1,050,112
│    └─OptLangLayer: 2-4                 524
│    │    └─LayerNorm: 3-7               1,024
│    │    └─Sequential: 3-8              1,050,112
├─Sequential: 1-5                        --
│    └─Linear: 2-5                       525,312
│    └─GELU: 2-6                         --
│    └─Line

In [3]:
import json
from tokenizers import Tokenizer
from torch.utils.data import Dataset, DataLoader

max_seq_len = 64
batch_size = 128

class Trans2019(Dataset):
    def __init__(self, filepath, tokenizer, seq_len=64):
        self.tokenizer = tokenizer
        self.seq_len = seq_len
        self.data_english = []
        self.data_chinese = []
        with open(filepath, "r", encoding="utf-8") as f:
            for line in f:
                datai = json.loads(line)
                self.data_english.append(datai['english'])
                self.data_chinese.append(datai['chinese'])
    
    def __getitem__(self, index):
        text_english = self.data_english[index]
        text_chinese = self.data_chinese[index]
        tokens_english = self.tokenizer.encode(text_english).ids
        tokens_chinese = [1] + self.tokenizer.encode(text_chinese).ids
        if len(tokens_english) >= self.seq_len:
            tokens_english = tokens_english[:self.seq_len]
        else:
            tokens_english = tokens_english + [3]*(self.seq_len - len(tokens_english))
        if len(tokens_chinese) >= self.seq_len:
            tokens_chinese = tokens_chinese[:self.seq_len]
        else:
            tokens_chinese = tokens_chinese + [2] + [3]*(self.seq_len - 1 - len(tokens_chinese))
        input_ids = torch.tensor(tokens_english, dtype=torch.long)
        target_ids = torch.tensor(tokens_chinese, dtype=torch.long)
        return input_ids, target_ids

    def __len__(self):
        return len(self.data_english)

tokenizer = Tokenizer.from_file("./cying/datasets/tokenizer-wiki.json")
trans_dataset = Trans2019(
    filepath=f'./cying/datasets/translation2019zh/translation2019zh_train.json',
    tokenizer=tokenizer,
    seq_len=max_seq_len
)
train_loader = DataLoader(
    dataset=trans_dataset,
    batch_size=batch_size,
    shuffle=True
)

In [ ]:
from tqdm import tqdm
epoch = 10**2
lr = 1e-3
optim = torch.optim.Adam(opt_lang_model.parameters(), lr=lr)
criterion = torch.nn.CrossEntropyLoss(ignore_index=3)

for i in range(epoch):
    pbar = tqdm(train_loader)
    for src_ids, tgt_ids in pbar:
        opt_lang_model.train()
        optim.zero_grad()
        outputs = opt_lang_model(
            src=src_ids.to(device),
            tgt=tgt_ids[:,:-1].to(device),
            src_padding_mask=(src_ids==3).to(device),
            tgt_padding_mask=(tgt_ids[:,:-1]==3).to(device),
            src_casual_mask=None,
            tgt_casual_mask=torch.triu(torch.ones(max_seq_len-1, max_seq_len-1), diagonal=1).bool().expand((src_ids.shape[0],max_seq_len-1,max_seq_len-1)).to(device)
        )
        loss = criterion(
            outputs.view(-1, outputs.size(-1)),
            tgt_ids[:,1:].reshape(-1).to(device)
        )
        loss.backward()
        optim.step()
        pbar.set_description(f"Epoch {i+1}/{epoch} Loss {loss.item():.4e}")

  0%|          | 0/40324 [00:00<?, ?it/s]/Users/wangh/Work/Code/PyCying/cying/nn/modules/opt_lang_layer.py:79: UserWarning: An output with one or more elements was resized since it had shape [], which does not match the required output shape [128, 64, 4, 65]. This behavior is deprecated, and in a future PyTorch release outputs will not be resized unless they have zero elements. You can explicitly reuse an out tensor t by resizing it, inplace, to zero elements with t.resize_(0). (Triggered internally at /Users/runner/work/pytorch/pytorch/pytorch/aten/src/ATen/native/Resize.cpp:38.)
  token_seq_fre = torch.fft.rfft(token_seq) / self.d_head
/Users/wangh/Work/Code/PyCying/cying/nn/modules/opt_lang_layer.py:79: UserWarning: An output with one or more elements was resized since it had shape [], which does not match the required output shape [128, 63, 4, 65]. This behavior is deprecated, and in a future PyTorch release outputs will not be resized unless they have zero elements. You can explic

In [58]:
def bytes_to_unicode():
    # 原始保留的字节范围
    bs = (
        list(range(ord("!"), ord("~") + 1)) +          # ASCII可打印字符（33-126）
        list(range(ord("¡"), ord("¬") + 1)) +          # 西班牙语特殊字符（161-172）
        list(range(ord("®"), ord("ÿ") + 1))            # 其他扩展字符（174-255）
    )
    
    cs = bs.copy()  # 初始字符列表
    n = 0
    
    # 遍历所有可能的字节（0-255）
    for b in range(2**8):
        if b not in bs:
            bs.append(b)
            cs.append(2**8 + n)  # 超出原始范围的字节映射到更高Unicode码位
            n += 1
    
    # 将码位转换为Unicode字符
    cs = [chr(code) for code in cs]
    
    return dict(zip(bs, cs))

def get_reverse_mapping(forward_map):
    """
    根据正向映射生成反向映射
    返回字典：{unicode_char: byte_value}
    """
    return {v: k for k, v in forward_map.items()}

# 生成映射表
forward_map = bytes_to_unicode()
reverse_map = get_reverse_mapping(forward_map)
from tokenizers import Tokenizer
tokenizer = Tokenizer.from_file("./cying/datasets/tokenizer-wiki.json")
max_seq_len = 64
device = torch.device('cpu')
def unicode_str_to_bytes(unicode_str):
    return bytes([reverse_map[c] for c in unicode_str])

In [ ]:
opt_lang_model.eval()
src = 'Good morning.'
tgt = torch.tensor([1] + [3]*63,dtype=torch.long).unsqueeze(0)
src = tokenizer.encode(src).ids
src = torch.tensor(src + [3]*(max_seq_len-len(src))).unsqueeze(0)
for i in range(63):
    tgt[0,i+1] = opt_lang_model(
        src.to(device),
        tgt[:,:-1].to(device),
        src.to(device)==3,
        tgt[:,:-1].to(device)==3,
        None,
        torch.triu(torch.ones(max_seq_len-1, max_seq_len-1), diagonal=1).bool().expand((1,max_seq_len-1,max_seq_len-1)).to(device)
    ).argmax(-1)[0,i].cpu()
    if tgt[0,i+1] == 2:
        break
unicode_str = tokenizer.decode(list(tgt[0])).replace(' ','')
# 将转换后的Unicode字符串还原为原始文本
recovered_bytes = unicode_str_to_bytes(unicode_str)
recovered_text = recovered_bytes.decode('utf-8')
print(f"还原后的文本：{recovered_text}")

还原后的文本：早上好。
